# BTS On-Time Performance: route profile (exploration)

Purpose: profile US domestic airport pairs so one route can be chosen to start the project with.
Exploration only: no modelling, no dbt models, and nothing in the data is cleaned or adjusted.
Anything that looks inconsistent is listed in section 3 rather than fixed.

**Source.** Bureau of Transportation Statistics (BTS), TranStats, *Marketing Carrier On-Time
Performance (Beginning January 2018)*, monthly pre-zipped CSV files:

`https://transtats.bts.gov/PREZIP/On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_<YYYY>_<M>.zip`

Table page: <https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGK>

**How the data got here.** `ingestion/bts_otp.py` downloads the files into `data/raw/bts_otp_marketing/`
and loads them into the DuckDB table `raw.bts_otp_marketing`, every column as text with the original
names, plus `_source_file`, `_source_year`, `_source_month`. One row per file is written to
`raw.bts_otp_load_log`.

**To re-run.**

```bash
uv run python ingestion/bts_otp.py
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/01_bts_otp_route_profile.ipynb
```

Every number below is produced by the SQL shown in the cell above it.

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DB = Path(os.environ.get("FLIGHT_DELAYS_DUCKDB_PATH", "data/flight_delays.duckdb"))
DB = DB if DB.is_absolute() else ROOT / DB
con = duckdb.connect(str(DB), read_only=True)
T = "raw.bts_otp_marketing"


def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()


print("database:", DB.relative_to(ROOT) if DB.is_relative_to(ROOT) else DB)
print("duckdb:", duckdb.__version__)

database: data\flight_delays.duckdb
duckdb: 1.5.6


## 1. Files and rows per month

First the load log written by the ingestion script, then the same months counted from the table itself.

In [2]:
load_log = q("""
    SELECT month, zip_bytes, remote_last_modified, csv_data_lines, rows_loaded, n_columns, status
    FROM raw.bts_otp_load_log
    ORDER BY month
""")
load_log

,month,zip_bytes,remote_last_modified,csv_data_lines,rows_loaded,n_columns,status
0,2023-01,30592621,"Tue, 28 Mar 2023 12:31:31 GMT",573877,573877,123,ok
1,2023-02,27954550,"Thu, 27 Apr 2023 11:59:20 GMT",536229,536229,123,ok
2,2023-03,32510491,"Tue, 23 May 2023 11:20:53 GMT",616234,616234,123,ok
3,2023-04,31511282,"Fri, 09 Jun 2023 17:55:00 GMT",596676,596676,123,ok
4,2023-05,31932462,"Thu, 07 Sep 2023 13:28:51 GMT",616630,616630,123,ok
5,2023-06,32626440,"Wed, 27 Sep 2023 12:42:29 GMT",613577,613577,123,ok
6,2023-07,33867598,"Tue, 17 Oct 2023 14:24:48 GMT",638995,638995,123,ok
7,2023-08,33134150,"Tue, 24 Oct 2023 13:24:02 GMT",640236,640236,123,ok
8,2023-09,31480933,"Fri, 01 Dec 2023 13:14:01 GMT",604715,604715,123,ok
9,2023-10,33025323,"Fri, 05 Jan 2024 14:04:42 GMT",635538,635538,123,ok


In [3]:
print(q("SELECT source_file, url FROM raw.bts_otp_load_log ORDER BY month LIMIT 1").T.to_string())
print()
print(
    q("""
    SELECT count(*) AS files,
           count(*) FILTER (WHERE status = 'ok') AS files_ok,
           count(*) FILTER (WHERE status <> 'ok') AS files_not_ok,
           count(*) FILTER (WHERE rows_loaded IS DISTINCT FROM csv_data_lines) AS files_row_count_differs,
           min(month) AS first_month, max(month) AS last_month,
           sum(rows_loaded) AS rows_loaded, sum(zip_bytes) AS zip_bytes
    FROM raw.bts_otp_load_log
""").T.to_string()
)

                                                                                                                            0
source_file                                   On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2023_1.zip
url          https://transtats.bts.gov/PREZIP/On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2023_1.zip

                                    0
files                              43
files_ok                           43
files_not_ok                        0
files_row_count_differs             0
first_month                   2023-01
last_month                    2026-07
rows_loaded                27117136.0
zip_bytes                1441186820.0


In [4]:
per_month = q(f"""
    WITH m AS (
        SELECT _source_year AS yr, _source_month AS mo,
               count(*) AS n_rows,
               count(DISTINCT FlightDate) AS distinct_flight_dates,
               day(last_day(make_date(_source_year, _source_month, 1))) AS days_in_month,
               min(FlightDate) AS min_flight_date,
               max(FlightDate) AS max_flight_date,
               count(*) FILTER (WHERE "Year" <> _source_year::VARCHAR
                                   OR "Month" <> _source_month::VARCHAR) AS rows_year_month_differs_from_file,
               count(*) FILTER (WHERE TRY_CAST(FlightDate AS DATE) IS NULL) AS rows_unparseable_flight_date,
               count(*) FILTER (WHERE year(TRY_CAST(FlightDate AS DATE)) <> _source_year
                                   OR month(TRY_CAST(FlightDate AS DATE)) <> _source_month) AS rows_flight_date_outside_month
        FROM {T}
        GROUP BY 1, 2
    )
    SELECT yr, mo, n_rows, distinct_flight_dates, days_in_month,
           round(n_rows / days_in_month) AS rows_per_day,
           round(100.0 * n_rows / lag(n_rows, 12) OVER (ORDER BY yr, mo) - 100, 1) AS pct_vs_same_month_prior_year,
           min_flight_date, max_flight_date,
           rows_year_month_differs_from_file, rows_unparseable_flight_date, rows_flight_date_outside_month
    FROM m
    ORDER BY yr, mo
""")
per_month

,yr,mo,n_rows,distinct_flight_dates,days_in_month,rows_per_day,pct_vs_same_month_prior_year,min_flight_date,max_flight_date,rows_year_month_differs_from_file,rows_unparseable_flight_date,rows_flight_date_outside_month
0,2023,1,573877,31,31,18512.0,NaN,2023-01-01,2023-01-31,0,0,0
1,2023,2,536229,28,28,19151.0,NaN,2023-02-01,2023-02-28,0,0,0
2,2023,3,616234,31,31,19879.0,NaN,2023-03-01,2023-03-31,0,0,0
3,2023,4,596676,30,30,19889.0,NaN,2023-04-01,2023-04-30,0,0,0
4,2023,5,616630,31,31,19891.0,NaN,2023-05-01,2023-05-31,0,0,0
5,2023,6,613577,30,30,20453.0,NaN,2023-06-01,2023-06-30,0,0,0
6,2023,7,638995,31,31,20613.0,NaN,2023-07-01,2023-07-31,0,0,0
7,2023,8,640236,31,31,20653.0,NaN,2023-08-01,2023-08-31,0,0,0
8,2023,9,604715,30,30,20157.0,NaN,2023-09-01,2023-09-30,0,0,0
9,2023,10,635538,31,31,20501.0,NaN,2023-10-01,2023-10-31,0,0,0


Months expected between the first and last month loaded but absent from the table, and months whose distinct flight dates differ from the days in the month:

In [5]:
print(
    q(f"""
    WITH bounds AS (SELECT min(make_date(_source_year, _source_month, 1)) AS d0,
                           max(make_date(_source_year, _source_month, 1)) AS d1 FROM {T}),
         expected AS (SELECT unnest(generate_series(d0, d1, INTERVAL 1 MONTH))::DATE AS m FROM bounds),
         present AS (SELECT DISTINCT make_date(_source_year, _source_month, 1) AS m FROM {T})
    SELECT (SELECT count(*) FROM expected) AS months_expected,
           (SELECT count(*) FROM present) AS months_present,
           (SELECT list(strftime(e.m, '%Y-%m')) FROM expected e ANTI JOIN present p USING (m)) AS months_missing
""").T.to_string()
)
print()
print("months where distinct flight dates != days in month:")
print(
    per_month[per_month.distinct_flight_dates != per_month.days_in_month][
        ["yr", "mo", "distinct_flight_dates", "days_in_month"]
    ].to_string(index=False)
)

                    0
months_expected    43
months_present     43
months_missing   <NA>

months where distinct flight dates != days in month:
Empty DataFrame
Columns: [yr, mo, distinct_flight_dates, days_in_month]
Index: []


The 15 days with the fewest rows, with each day's share of the median daily row count of its own month (to spot partial days):

In [6]:
q(f"""
    WITH d AS (SELECT FlightDate, _source_year AS yr, _source_month AS mo, count(*) AS n_rows,
                      count(*) FILTER (WHERE Cancelled = '1.00') AS n_cancelled
               FROM {T} GROUP BY 1, 2, 3),
         x AS (SELECT *, median(n_rows) OVER (PARTITION BY yr, mo) AS month_median FROM d)
    SELECT FlightDate, dayname(FlightDate::DATE) AS weekday, n_rows, n_cancelled,
           month_median, round(100.0 * n_rows / month_median, 1) AS pct_of_month_median
    FROM x ORDER BY n_rows LIMIT 15
""")

,FlightDate,weekday,n_rows,n_cancelled,month_median,pct_of_month_median
0,2024-11-28,Thursday,11929,2,21421.0,55.7
1,2025-11-27,Thursday,11963,18,21990.0,54.4
2,2023-11-23,Thursday,12052,4,20738.5,58.1
3,2023-01-14,Saturday,15844,34,18456.0,85.8
4,2025-01-18,Saturday,16273,11,19988.0,81.4
5,2023-01-21,Saturday,16414,25,18456.0,88.9
6,2023-01-28,Saturday,16457,229,18456.0,89.2
7,2024-01-13,Saturday,16501,1125,19251.0,85.7
8,2024-02-03,Saturday,16546,246,19524.0,84.7
9,2024-01-20,Saturday,16580,320,19251.0,86.1


## 2. Every column and its null rate

All columns of the source file, in file order. A value counts as null when the CSV field is empty.
The three `_source_*` columns are added by the ingestion script and are listed last.

In [7]:
cols = [r[0] for r in con.sql(f"DESCRIBE {T}").fetchall()]
null_sql = (
    "SELECT count(*) AS n_rows, " + ", ".join(f'count("{c}") AS "{c}"' for c in cols) + f" FROM {T}"
)
print(null_sql[:400], "...", f"[{len(cols)} count() expressions, one per column]")
counts = con.sql(null_sql).df().iloc[0]
n_rows = int(counts["n_rows"])
nulls = pd.DataFrame(
    {
        "position": range(1, len(cols) + 1),
        "column": cols,
        "non_null": [int(counts[c]) for c in cols],
    }
)
nulls["nulls"] = n_rows - nulls["non_null"]
nulls["null_pct"] = (100 * nulls["nulls"] / n_rows).round(3)
print("rows:", n_rows, "| columns:", len(cols))
with pd.option_context("display.max_rows", None):
    display(nulls.set_index("position"))

SELECT count(*) AS n_rows, count("Year") AS "Year", count("Quarter") AS "Quarter", count("Month") AS "Month", count("DayofMonth") AS "DayofMonth", count("DayOfWeek") AS "DayOfWeek", count("FlightDate") AS "FlightDate", count("Marketing_Airline_Network") AS "Marketing_Airline_Network", count("Operated_or_Branded_Code_Share_Partners") AS "Operated_or_Branded_Code_Share_Partners", count("DOT_ID_Marke ... [123 count() expressions, one per column]


rows: 27117136 | columns: 123


,column,non_null,nulls,null_pct
position,,,,
1,Year,27117136,0,0.000
2,Quarter,27117136,0,0.000
3,Month,27117136,0,0.000
4,DayofMonth,27117136,0,0.000
5,DayOfWeek,27117136,0,0.000
6,FlightDate,27117136,0,0.000
7,Marketing_Airline_Network,27117136,0,0.000
8,Operated_or_Branded_Code_Share_Partners,27117136,0,0.000
9,DOT_ID_Marketing_Airline,27117136,0,0.000


## 3. Consistency checks

Nothing here is corrected. Each line is a count of rows (or groups) matching a condition, so anything
non-zero that should be zero is visible.

In [8]:
for col in [
    "Cancelled",
    "Diverted",
    "Flights",
    "Duplicate",
    "CancellationCode",
    "DivAirportLandings",
]:
    print(
        q(f'SELECT "{col}", count(*) AS n_rows FROM {T} GROUP BY 1 ORDER BY 2 DESC').to_string(
            index=False
        )
    )
    print()

Cancelled   n_rows
     0.00 26696728
     1.00   420408

Diverted   n_rows
    0.00 27045094
    1.00    72042

Flights   n_rows
   1.00 27117136

Duplicate   n_rows
        N 27117136



CancellationCode   n_rows
             NaN 26696728
               B   254141
               A   101458
               C    64461
               D      348

DivAirportLandings   n_rows
                 0 27042146
                 1    71424
                 9     2941
                 2      611
               NaN        7
                 3        7



In [9]:
con.sql(f"""
    CREATE OR REPLACE TEMP VIEW otp AS
    SELECT TRY_CAST(FlightDate AS DATE) AS flight_date,
           Marketing_Airline_Network AS mkt, Operating_Airline AS op,
           Flight_Number_Marketing_Airline AS flt, Tail_Number AS tail,
           Origin AS origin, Dest AS dest, OriginAirportID AS origin_id, DestAirportID AS dest_id,
           OriginState AS origin_state, DestState AS dest_state,
           CRSDepTime AS crs_dep, DepTime AS dep_time, ArrTime AS arr_time,
           TRY_CAST(DepDelay AS DOUBLE) AS dep_delay,
           TRY_CAST(DepDelayMinutes AS DOUBLE) AS dep_delay_min,
           TRY_CAST(DepDel15 AS DOUBLE) AS dep_del15,
           TRY_CAST(ArrDelay AS DOUBLE) AS arr_delay,
           TRY_CAST(ArrDelayMinutes AS DOUBLE) AS arr_delay_min,
           TRY_CAST(ArrDel15 AS DOUBLE) AS arr_del15,
           TRY_CAST(Cancelled AS DOUBLE) AS cancelled,
           TRY_CAST(Diverted AS DOUBLE) AS diverted,
           TRY_CAST(Flights AS DOUBLE) AS flights,
           TRY_CAST(CarrierDelay AS DOUBLE) AS carrier_delay,
           TRY_CAST(WeatherDelay AS DOUBLE) AS weather_delay,
           TRY_CAST(NASDelay AS DOUBLE) AS nas_delay,
           TRY_CAST(SecurityDelay AS DOUBLE) AS security_delay,
           TRY_CAST(LateAircraftDelay AS DOUBLE) AS late_aircraft_delay
    FROM {T}
""")
checks = q("""
    SELECT
        count(*) AS rows_total,
        count(*) FILTER (WHERE cancelled IS NULL) AS cancelled_not_numeric_or_null,
        count(*) FILTER (WHERE diverted IS NULL) AS diverted_not_numeric_or_null,
        count(*) FILTER (WHERE cancelled = 1 AND diverted = 1) AS cancelled_and_diverted,
        count(*) FILTER (WHERE cancelled = 1 AND dep_time IS NOT NULL) AS cancelled_with_departure_time,
        count(*) FILTER (WHERE cancelled = 1 AND arr_time IS NOT NULL) AS cancelled_with_arrival_time,
        count(*) FILTER (WHERE cancelled = 0 AND dep_del15 IS NULL) AS not_cancelled_dep_del15_null,
        count(*) FILTER (WHERE cancelled = 0 AND diverted = 0 AND arr_del15 IS NULL) AS not_cancelled_not_diverted_arr_del15_null,
        count(*) FILTER (WHERE diverted = 1 AND arr_del15 IS NOT NULL) AS diverted_with_arr_del15,
        count(*) FILTER (WHERE dep_del15 = 1 AND dep_delay_min < 15) AS dep_del15_flag_but_under_15_min,
        count(*) FILTER (WHERE dep_del15 = 0 AND dep_delay_min >= 15) AS dep_del15_zero_but_15_plus_min,
        count(*) FILTER (WHERE arr_del15 = 1 AND arr_delay_min < 15) AS arr_del15_flag_but_under_15_min,
        count(*) FILTER (WHERE arr_del15 = 0 AND arr_delay_min >= 15) AS arr_del15_zero_but_15_plus_min,
        count(*) FILTER (WHERE arr_del15 = 1 AND carrier_delay IS NULL) AS arr_delayed_15_without_cause_minutes,
        count(*) FILTER (WHERE carrier_delay IS NOT NULL AND coalesce(arr_del15, 0) <> 1) AS cause_minutes_without_arr_del15,
        count(*) FILTER (WHERE carrier_delay IS NOT NULL
                           AND carrier_delay + weather_delay + nas_delay + security_delay + late_aircraft_delay
                               <> arr_delay_min) AS cause_minutes_sum_differs_from_arr_delay,
        count(*) FILTER (WHERE (carrier_delay IS NULL) <> (weather_delay IS NULL)
                            OR (carrier_delay IS NULL) <> (nas_delay IS NULL)
                            OR (carrier_delay IS NULL) <> (security_delay IS NULL)
                            OR (carrier_delay IS NULL) <> (late_aircraft_delay IS NULL)) AS cause_columns_partly_null,
        count(*) FILTER (WHERE origin = dest) AS origin_equals_dest,
        count(*) FILTER (WHERE flights <> 1 OR flights IS NULL) AS flights_column_not_1,
        count(*) FILTER (WHERE tail IS NULL) AS tail_number_null,
        count(*) FILTER (WHERE tail IS NULL AND cancelled = 0) AS tail_number_null_not_cancelled,
        count(*) FILTER (WHERE dep_delay > 1440) AS dep_delay_over_24h,
        max(dep_delay) AS max_dep_delay_min, min(dep_delay) AS min_dep_delay_min,
        max(arr_delay) AS max_arr_delay_min, min(arr_delay) AS min_arr_delay_min
    FROM otp
""").T
checks.columns = ["value"]
checks

,value
rows_total,27117136.0
cancelled_not_numeric_or_null,0.0
diverted_not_numeric_or_null,0.0
cancelled_and_diverted,0.0
cancelled_with_departure_time,17380.0
cancelled_with_arrival_time,0.0
not_cancelled_dep_del15_null,0.0
not_cancelled_not_diverted_arr_del15_null,2.0
diverted_with_arr_del15,0.0
dep_del15_flag_but_under_15_min,0.0


In [10]:
print(
    q("""
    WITH k AS (SELECT flight_date, mkt, flt, origin, dest, crs_dep, count(*) AS n
               FROM otp GROUP BY ALL HAVING count(*) > 1)
    SELECT count(*) AS duplicate_key_groups, coalesce(sum(n), 0) AS rows_in_those_groups,
           coalesce(sum(n) - count(*), 0) AS extra_rows
    FROM k
""").T.to_string()
)
print(
    "(key: flight date, marketing airline, marketing flight number, origin, destination, scheduled departure time)"
)
print()
print(
    q("""
    SELECT count(*) FILTER (WHERE n_ids > 1) AS airport_codes_with_more_than_one_airport_id,
           count(*) AS airport_codes
    FROM (SELECT origin, count(DISTINCT origin_id) AS n_ids FROM otp GROUP BY 1)
""").T.to_string()
)
print()
print("rows by origin state outside the 50 states and DC:")
print(
    q("""
    SELECT origin_state, count(*) AS n_rows FROM otp
    WHERE origin_state NOT IN ('AL','AK','AZ','AR','CA','CO','CT','DE','DC','FL','GA','HI','ID','IL','IN','IA','KS','KY',
        'LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI',
        'SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY') OR origin_state IS NULL
    GROUP BY 1 ORDER BY 2 DESC
""").to_string(index=False)
)

                        0
duplicate_key_groups  0.0
rows_in_those_groups  0.0
extra_rows            0.0
(key: flight date, marketing airline, marketing flight number, origin, destination, scheduled departure time)

                                               0
airport_codes_with_more_than_one_airport_id    0
airport_codes                                374

rows by origin state outside the 50 states and DC:


origin_state  n_rows
          PR  137834
          VI   23798
          TT    4429


## 4. Top 30 airport pairs by flight volume (each direction separately)

Definitions used in the three tables below:

- **Pair**: `Origin` → `Dest` airport codes; each direction is its own row. Ranked by row count.
- **Flights**: number of rows (one row is one scheduled flight; `sum(Flights)` is shown alongside as a check).
- **Average flights per day**: flights divided by the number of calendar days from the first to the last
  `FlightDate` in the whole table. `days_with_flights` shows how many of those days the pair actually operated.
- **Carriers**: distinct `Marketing_Airline_Network` and distinct `Operating_Airline` codes.
- **% delayed 15+**: `DepDel15` / `ArrDel15` equal to 1, as a share of flights where the flag is not null
  (cancelled flights have no flag; diverted flights have no arrival flag). The share of all scheduled
  flights is shown next to it.
- **% cancelled, % diverted, % null tail number**: share of all rows of the pair.
- **Delay minutes by cause**: sum of each cause column. BTS fills these only for flights that arrived
  15 or more minutes late; percentages are of the five causes' total.

In [11]:
TOP_SQL = """
    WITH period AS (
        SELECT min(flight_date) AS d0, max(flight_date) AS d1,
               datediff('day', min(flight_date), max(flight_date)) + 1 AS n_days
        FROM otp
    ),
    pairs AS (
        SELECT origin, dest,
               count(*) AS flights,
               sum(flights) AS sum_flights_column,
               count(DISTINCT flight_date) AS days_with_flights,
               count(DISTINCT mkt) AS n_marketing_carriers,
               count(DISTINCT op) AS n_operating_carriers,
               string_agg(DISTINCT mkt, ' ' ORDER BY mkt) AS marketing_carriers,
               string_agg(DISTINCT op, ' ' ORDER BY op) AS operating_carriers,
               count(dep_del15) AS dep_del15_reported,
               sum(dep_del15) AS dep_delayed_15,
               count(arr_del15) AS arr_del15_reported,
               sum(arr_del15) AS arr_delayed_15,
               sum(cancelled) AS cancelled,
               sum(diverted) AS diverted,
               count(*) FILTER (WHERE tail IS NULL) AS tail_null,
               coalesce(sum(carrier_delay), 0) AS carrier_min,
               coalesce(sum(weather_delay), 0) AS weather_min,
               coalesce(sum(nas_delay), 0) AS nas_min,
               coalesce(sum(security_delay), 0) AS security_min,
               coalesce(sum(late_aircraft_delay), 0) AS late_aircraft_min
        FROM otp
        GROUP BY origin, dest
    )
    SELECT row_number() OVER (ORDER BY flights DESC, origin, dest) AS rank, p.*, period.n_days,
           carrier_min + weather_min + nas_min + security_min + late_aircraft_min AS cause_min_total
    FROM pairs p, period
    QUALIFY rank <= 30
    ORDER BY rank
"""
top = q(TOP_SQL)
print(
    q(
        "SELECT min(flight_date) AS first_flight_date, max(flight_date) AS last_flight_date, "
        "datediff('day', min(flight_date), max(flight_date)) + 1 AS n_days, count(*) AS rows_total, "
        "count(DISTINCT origin || '-' || dest) AS directional_pairs FROM otp"
    ).T.to_string()
)
print("top 30 share of all rows: %.2f%%" % (100 * top.flights.sum() / n_rows))

                                     0
first_flight_date  2023-01-01 00:00:00
last_flight_date   2026-07-31 00:00:00
n_days                            1308
rows_total                    27117136
directional_pairs                 7965
top 30 share of all rows: 3.62%


### 4a. Volume and carriers

In [12]:
a = top[
    ["rank", "origin", "dest", "flights", "sum_flights_column", "n_days", "days_with_flights"]
].copy()
a.insert(4, "avg_flights_per_day", (top.flights / top.n_days).round(2))
a["n_marketing_carriers"] = top.n_marketing_carriers
a["n_operating_carriers"] = top.n_operating_carriers
a["marketing_carriers"] = top.marketing_carriers
a["operating_carriers"] = top.operating_carriers
a.set_index("rank")

,origin,dest,flights,avg_flights_per_day,sum_flights_column,n_days,days_with_flights,n_marketing_carriers,n_operating_carriers,marketing_carriers,operating_carriers
rank,,,,,,,,,,,
1,LAX,SFO,42204,32.27,42204.0,1308,1308,7,9,AA AS B6 DL F9 UA WN,AA AS B6 DL F9 OO QX UA WN
2,SFO,LAX,42141,32.22,42141.0,1308,1308,7,10,AA AS B6 DL F9 UA WN,AA AS B6 DL F9 MQ OO QX UA WN
3,OGG,HNL,40688,31.11,40688.0,1308,1308,3,3,AS HA WN,AS HA WN
4,HNL,OGG,40681,31.10,40681.0,1308,1308,3,3,AS HA WN,AS HA WN
5,LGA,ORD,39202,29.97,39202.0,1308,1308,4,7,AA DL NK UA,9E AA DL NK OO UA YX
6,ORD,LGA,39200,29.97,39200.0,1308,1308,4,7,AA DL NK UA,9E AA DL NK OO UA YX
7,JFK,LAX,36087,27.59,36087.0,1308,1308,4,4,AA B6 DL F9,AA B6 DL F9
8,LAX,JFK,36086,27.59,36086.0,1308,1308,4,4,AA B6 DL F9,AA B6 DL F9
9,LAX,LAS,35707,27.30,35707.0,1308,1308,8,10,AA AS B6 DL F9 NK UA WN,AA AS B6 DL F9 NK OO QX UA WN


### 4b. Delays, cancellations, diversions, tail numbers

In [13]:
b = top[["rank", "origin", "dest", "flights"]].copy()
b["pct_dep_delayed_15_of_reported"] = (100 * top.dep_delayed_15 / top.dep_del15_reported).round(2)
b["pct_arr_delayed_15_of_reported"] = (100 * top.arr_delayed_15 / top.arr_del15_reported).round(2)
b["pct_dep_delayed_15_of_scheduled"] = (100 * top.dep_delayed_15 / top.flights).round(2)
b["pct_arr_delayed_15_of_scheduled"] = (100 * top.arr_delayed_15 / top.flights).round(2)
b["pct_cancelled"] = (100 * top.cancelled / top.flights).round(2)
b["pct_diverted"] = (100 * top.diverted / top.flights).round(2)
b["tail_null_rows"] = top.tail_null
b["pct_tail_null"] = (100 * top.tail_null / top.flights).round(3)
b.set_index("rank")

,origin,dest,flights,pct_dep_delayed_15_of_reported,pct_arr_delayed_15_of_reported,pct_dep_delayed_15_of_scheduled,pct_arr_delayed_15_of_scheduled,pct_cancelled,pct_diverted,tail_null_rows,pct_tail_null
rank,,,,,,,,,,,
1,LAX,SFO,42204,24.33,30.11,24.14,29.81,0.81,0.18,110,0.261
2,SFO,LAX,42141,23.42,22.87,23.22,22.66,0.90,0.04,115,0.273
3,OGG,HNL,40688,12.78,15.55,12.63,15.36,1.20,0.02,336,0.826
4,HNL,OGG,40681,12.00,13.76,11.88,13.62,1.03,0.03,286,0.703
5,LGA,ORD,39202,20.68,24.35,20.05,23.48,3.30,0.27,428,1.092
6,ORD,LGA,39200,24.41,26.30,23.69,25.33,3.16,0.53,391,0.997
7,JFK,LAX,36087,17.40,20.63,17.24,20.37,1.11,0.18,3,0.008
8,LAX,JFK,36086,20.66,20.27,20.42,19.92,1.19,0.55,1,0.003
9,LAX,LAS,35707,23.58,24.35,23.36,24.11,0.94,0.04,71,0.199


### 4c. Delay minutes by cause

In [14]:
c = top[
    [
        "rank",
        "origin",
        "dest",
        "carrier_min",
        "weather_min",
        "nas_min",
        "security_min",
        "late_aircraft_min",
        "cause_min_total",
    ]
].copy()
for name in ["carrier", "weather", "nas", "security", "late_aircraft"]:
    c[f"pct_{name}"] = (100 * top[f"{name}_min"] / top.cause_min_total).round(1)
for name in [
    "carrier_min",
    "weather_min",
    "nas_min",
    "security_min",
    "late_aircraft_min",
    "cause_min_total",
]:
    c[name] = c[name].astype("int64")
c.set_index("rank")

,origin,dest,carrier_min,weather_min,nas_min,security_min,late_aircraft_min,cause_min_total,pct_carrier,pct_weather,pct_nas,pct_security,pct_late_aircraft
rank,,,,,,,,,,,,,
1,LAX,SFO,167310,10959,336288,483,220786,735826,22.7,1.5,45.7,0.1,30.0
2,SFO,LAX,155795,8115,92990,388,338527,595815,26.1,1.4,15.6,0.1,56.8
3,OGG,HNL,71889,7229,33019,395,150635,263167,27.3,2.7,12.5,0.2,57.2
4,HNL,OGG,104594,5876,11743,565,93020,215798,48.5,2.7,5.4,0.3,43.1
5,LGA,ORD,199218,46222,238821,493,328661,813415,24.5,5.7,29.4,0.1,40.4
6,ORD,LGA,208328,34783,255676,578,303249,802614,26.0,4.3,31.9,0.1,37.8
7,JFK,LAX,241842,17682,146635,726,144128,551013,43.9,3.2,26.6,0.1,26.2
8,LAX,JFK,279423,3647,147822,1110,229368,661370,42.2,0.6,22.4,0.2,34.7
9,LAX,LAS,136580,7152,160464,920,240635,545751,25.0,1.3,29.4,0.2,44.1


### 4d. Checks on the top 30

In [15]:
print("pairs where count(*) != sum(Flights):", int((top.flights != top.sum_flights_column).sum()))
print(
    "pairs not operated on every day of the period:",
    int((top.days_with_flights != top.n_days).sum()),
)
rev = top.merge(top, left_on=["origin", "dest"], right_on=["dest", "origin"], suffixes=("", "_rev"))
print("pairs whose reverse direction is also in the top 30:", len(rev), "of", len(top))
print("top-30 pairs whose reverse direction is NOT in the top 30:")
print(
    top[~top.set_index(["origin", "dest"]).index.isin(rev.set_index(["origin", "dest"]).index)][
        ["rank", "origin", "dest", "flights"]
    ].to_string(index=False)
)

pairs where count(*) != sum(Flights): 0
pairs not operated on every day of the period: 0
pairs whose reverse direction is also in the top 30: 30 of 30
top-30 pairs whose reverse direction is NOT in the top 30:
Empty DataFrame
Columns: [rank, origin, dest, flights]
Index: []


In [16]:
con.close()